In [1]:
import tkinter as tk
from tkinter import filedialog, ttk
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import numpy as np

In [2]:
class LinearModel:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def plot(self, ax=None):
        line_model = stats.linregress(self.x, self.y)
        slope = line_model[0]
        intercept = line_model[1]
        if ax is None:
            ax = plt.gca()
        ax.scatter(self.x, self.y)
        
        fit_line = slope * self.x + intercept
        ax.plot(self.x, fit_line, label="best fit line", color='red')
        
        ax.set_title(f"Slope: {slope:.2f}")
        ax.legend()


In [3]:
class DataAnalysisApp:
    def __init__(self, root):
        self.root = root
        self.root.title("FCS FITTING APP")
        self.root.geometry("1200x800")
        
        # Internal State
        self.loaded_files = []
        
        self.setup_ui()

    def setup_ui(self):
        # --- Main Layout Containers ---
        self.sidebar = tk.Frame(self.root, width=250, bg="#f0f0f0", relief="sunken", borderwidth=1)
        self.sidebar.pack(side="left", fill="y")
        
        self.main_content = tk.Frame(self.root, bg="white")
        self.main_content.pack(side="right", expand=True, fill="both")

        # --- Sidebar Components ---
        tk.Label(self.sidebar, text="File Explorer", font=("Arial", 12, "bold"), bg="#f0f0f0").pack(pady=10)
        
        btn_frame = tk.Frame(self.sidebar, bg="#f0f0f0")
        btn_frame.pack(fill="x", padx=5)
        
        tk.Button(btn_frame, text="Load Files", command=self.load_files).pack(side="left", expand=True, fill="x")
        tk.Button(btn_frame, text="Load Folder", command=self.load_folder).pack(side="left", expand=True, fill="x")

        self.file_listbox = tk.Listbox(self.sidebar, selectmode="single")
        self.file_listbox.pack(expand=True, fill="both", padx=5, pady=5)
        self.file_listbox.bind("<<ListboxSelect>>", self.on_file_select)

        # --- Top Control Bar (Fitting Options) ---
        self.controls = tk.Frame(self.main_content, height=100, relief="raised", borderwidth=1)
        self.controls.pack(side="top", fill="x")

        tk.Label(self.controls, text="Fitting Style:").grid(row=0, column=0, padx=5, pady=10)
        self.fit_style = ttk.Combobox(self.controls, values=["Linear", "Exponential", "Gaussian", "Diffusion"])
        self.fit_style.set("Linear")
        self.fit_style.grid(row=0, column=1, padx=5)

        tk.Label(self.controls, text="Parameters:").grid(row=0, column=2, padx=5)
        self.param_entry = tk.Entry(self.controls)
        self.param_entry.insert(0, "p1=1.0, p2=0.5")
        self.param_entry.grid(row=0, column=3, padx=5)

        tk.Button(self.controls, text="Update Fit", command=self.update_graphs, bg="#e1e1e1").grid(row=0, column=4, padx=10)
        # --- Graphing Area (2x2 Grid) ---
        self.fig, self.axs = plt.subplots(3, 1, figsize=(8, 6), constrained_layout=True)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.main_content)
        self.canvas.get_tk_widget().pack(expand=True, fill="both")
        
        # Initialize subplots
        self.axs[0].set_title(f"Fit: {self.fit_style.get()}")
        self.axs[1].set_title("Intensity Trace")
        self.axs[2].set_title("Residuals")

    def load_files(self):
        files = filedialog.askopenfilenames(
            filetypes=[("Data files", "*.dat *.czi *.ptu"), ("All files", "*.*")]
        )
        if files:
            for f in files:
                if f not in self.loaded_files:
                    self.loaded_files.append(f)
                    self.file_listbox.insert(tk.END, f.split("/")[-1])

    def load_folder(self):
        folder = filedialog.askdirectory()
        if folder:
            import os
            extensions = ('.dat', '.czi', '.ptu')
            for root, dirs, files in os.walk(folder):
                for file in files:
                    if file.endswith(extensions):
                        full_path = os.path.join(root, file)
                        if full_path not in self.loaded_files:
                            self.loaded_files.append(full_path)
                            self.file_listbox.insert(tk.END, file)
    
    def dat_to_df(self,dat):
        df = pd.read_csv(dat, sep='\s+', skiprows=1)
        print(df)
        return df

    def on_file_select(self, event):
        selection = event.widget.curselection()
        if selection:
            index = selection[0]
            selected_file = self.loaded_files[index]
            print(f"Loading: {selected_file}")
            df = self.dat_to_df(selected_file)
            self.update_graphs(df)

    def update_graphs(self, df: pd.DataFrame):
        selected_style = self.fit_style.get()
       
        for ax in self.axs.flat:
            ax.clear()
        if selected_style == "Linear":
            model = LinearModel(np.log(df.iloc[:,0]),df.iloc[:, 1])
            model.plot(ax=self.axs[0])
            
        self.axs[0].set_title(f"Fit: {self.fit_style.get()}")
        self.axs[1].set_title("Intensity Trace")
        self.axs[2].set_title("Residuals")
  
        
        self.canvas.draw()
    

<>:82: SyntaxWarning: invalid escape sequence '\s'
<>:82: SyntaxWarning: invalid escape sequence '\s'
C:\Users\hanni\AppData\Local\Temp\ipykernel_76612\4008099963.py:82: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(dat, sep='\s+', skiprows=1)


In [ ]:
root = tk.Tk()
app = DataAnalysisApp(root)
root.mainloop()

Loading: C:/Users/hanni/Downloads/dat-20260505T073325Z-3-001/dat\atto5651nM_10CP.dat
     Correlation_time_ms       G_t  Err
0               0.000002  1.500635    0
1               0.000004  1.352390    0
2               0.000005  1.289709    0
3               0.000006  1.152865    0
4               0.000007  1.165952    0
..                   ...       ...  ...
109            15.099494 -0.000246    0
110            17.616077 -0.000170    0
111            20.132659  0.000635    0
112            22.649242  0.000111    0
113            25.165824 -0.000499    0

[114 rows x 3 columns]
Loading: C:/Users/hanni/Downloads/dat-20260505T073325Z-3-001/dat\atto5651nM_1CP.dat
     Correlation_time_ms       G_t  Err
0               0.000002  1.689190    0
1               0.000004  1.501902    0
2               0.000005  1.407656    0
3               0.000006  1.288475    0
4               0.000007  1.258004    0
..                   ...       ...  ...
109            15.099494 -0.000014    0
110    

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 2079, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
TypeError: DataAnalysisApp.update_graphs() missing 1 required positional argument: 'df'


Loading: C:/Users/hanni/Downloads/dat-20260505T073325Z-3-001/dat\atto5651nM_purewater.dat
     Correlation_time_ms       G_t  Err
0               0.000002  2.953070    0
1               0.000004  2.674995    0
2               0.000005  2.534386    0
3               0.000006  2.376987    0
4               0.000007  2.331066    0
..                   ...       ...  ...
109            15.099494 -0.000275    0
110            17.616077 -0.000195    0
111            20.132659 -0.000312    0
112            22.649242 -0.000403    0
113            25.165824 -0.000376    0

[114 rows x 3 columns]
Loading: C:/Users/hanni/Downloads/dat-20260505T073325Z-3-001/dat\atto5651nM_5CP.dat
     Correlation_time_ms       G_t  Err
0               0.000002  0.866589    0
1               0.000004  0.811681    0
2               0.000005  0.729588    0
3               0.000006  0.708044    0
4               0.000007  0.706130    0
..                   ...       ...  ...
109            15.099494 -0.000132    0
11

In [2]:
%pip install multipletau aicspylibczi ptufile

   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ------------------ --------------------- 262.1/559.5 kB ? eta -:--:--
   ---------------------------------------- 559.5/559.5 kB 1.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [ptufile]
   ------------- -------------------------- 1/3 [multipletau]
   -------------------------- ------------- 2/3 [aicspylibczi]
   ---------------------------------------- 3/3 [aicspylibczi]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
